# 02 — Event Exploration

**Milestone 1A decision-point figure.**

Before asking any model to discover a regime, look at what the underlying
state trajectories actually look like across all cataloged events.

Prerequisites:
- `python -m gridregime.pipeline.milestone_1a` has been run successfully
- `data/processed/event_windows.parquet` exists
- `results/tables/event_data_coverage.csv` shows <5% missing for all events

This notebook does **no modelling**. Output: `results/figures/event_survey.png`.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 120

ROOT = Path('..').resolve()

## 1. Coverage gate

Fail fast if any event has significant missing data.

In [ ]:
from gridregime.evaluation.coverage import assert_coverage, load_coverage

coverage = load_coverage(ROOT / 'results/tables/event_data_coverage.csv')
display(coverage)

assert_coverage(coverage, max_missing_pct=5.0)
print('Coverage gate passed.')

## 2. Load event windows

In [ ]:
windows = pd.read_parquet(ROOT / 'data/processed/event_windows.parquet')
events  = pd.read_csv(ROOT / 'metadata/events.csv')

print(f"{len(windows):,} rows across {windows['event_id'].nunique()} events")
windows.head()

## 3. Event survey figure

4 panels per event (demand, forecast error, ramp, interchange), all aligned
on `hours_to_event`.

- **Red (●)**: positive events — operator action taken
- **Blue (○)**: hard-negative controls — stress present, no action
- **Dashed vertical**: t = 0 (operator_action_time for positives; window_start for controls)

In [ ]:
from gridregime.plotting.event_survey import plot_event_survey, save_survey

fig = plot_event_survey(windows, events, ncols=3)
save_survey(fig, ROOT / 'results/figures')
plt.show()

## 4. Per-event quick-look

Drill into individual events here as needed before moving to Milestone 1B.

In [ ]:
from gridregime.plotting.event_survey import plot_event_survey

EVENT_ID = 'E2022_ELL_23'  # change as needed

fig = plot_event_survey(windows, events, event_ids=[EVENT_ID], ncols=1)
plt.show()